In [0]:

from pyspark.sql import functions as F
from src.analytics.gold import (
    sales_by_year,
    sales_by_platform,
    sales_by_genre,
    regional_sales,
    top_selling_games
)

# --------------------------------------------------
# 1. CONFIGURATION
# --------------------------------------------------

source_table = "workspace.video_game_sales.silver_video_game_sales"

gold_year_table = "workspace.video_game_sales.gold_sales_by_year"
gold_platform_table = "workspace.video_game_sales.gold_sales_by_platform"
gold_genre_table = "workspace.video_game_sales.gold_sales_by_genre"
gold_region_table = "workspace.video_game_sales.gold_sales_by_region"
gold_top_games_table = "workspace.video_game_sales.gold_top_games"

# Read Silver layer
silver_df = spark.table(source_table)

source_count = silver_df.count()

print(f"Silver source records: {source_count}")


# --------------------------------------------------
# 2. GOLD: SALES BY YEAR
# --------------------------------------------------

# Preserve records with invalid/missing years under UNKNOWN
year_df = (
    sales_by_year(silver_df)
    .withColumn("pipeline_layer", F.lit("GOLD"))
    .withColumn("created_at", F.current_timestamp())
)

year_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(gold_year_table)


# --------------------------------------------------
# 3. GOLD: SALES BY PLATFORM
# --------------------------------------------------

platform_df = (
    sales_by_platform(silver_df)
    .withColumn("pipeline_layer", F.lit("GOLD"))
    .withColumn("created_at", F.current_timestamp())
)

platform_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(gold_platform_table)


# --------------------------------------------------
# 4. GOLD: SALES BY GENRE
# --------------------------------------------------

genre_df = (
    sales_by_genre(silver_df)
    .withColumn("pipeline_layer", F.lit("GOLD"))
    .withColumn("created_at", F.current_timestamp())
)

genre_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(gold_genre_table)


# --------------------------------------------------
# 5. GOLD: REGIONAL SALES PERFORMANCE
# --------------------------------------------------

region_df = (
    regional_sales(silver_df)
    .withColumn("pipeline_layer", F.lit("GOLD"))
    .withColumn("created_at", F.current_timestamp())
)

region_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(gold_region_table)


# --------------------------------------------------
# 6. GOLD: TOP 10 BEST-SELLING GAMES
# --------------------------------------------------

top_games_df = (
    top_selling_games(silver_df, limit=10)
    .withColumn("pipeline_layer", F.lit("GOLD"))
    .withColumn("created_at", F.current_timestamp())
)

top_games_df.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(gold_top_games_table)


# --------------------------------------------------
# 7. VALIDATION
# --------------------------------------------------

gold_tables = {
    "YEAR": gold_year_table,
    "PLATFORM": gold_platform_table,
    "GENRE": gold_genre_table,
    "REGION": gold_region_table,
    "TOP_GAMES": gold_top_games_table
}

print("\nGOLD TABLE VALIDATION")

for name, table_name in gold_tables.items():

    result_df = spark.table(table_name)
    count = result_df.count()

    print(f"{name}: {count} records | {table_name}")

# Validate that the year, platform and genre
# aggregations preserve all source records.
for name, table_name in [
    ("YEAR", gold_year_table),
    ("PLATFORM", gold_platform_table),
    ("GENRE", gold_genre_table)
]:

    aggregated_count = (
        spark.table(table_name)
        .agg(F.sum("game_count").alias("total"))
        .first()["total"]
    )

    assert aggregated_count == source_count, (
        f"{name} aggregation count mismatch: "
        f"{aggregated_count} vs {source_count}"
    )

# Validate regional aggregation
regional_count = (
    spark.table(gold_region_table)
    .agg(F.sum("game_count").alias("total"))
    .first()["total"]
)

assert regional_count == source_count * 4

# Validate top games
top_count = spark.table(gold_top_games_table).count()
assert top_count == 10

# --------------------------------------------------
# 8. SALES RECONCILIATION
# --------------------------------------------------

source_global_sales = (
    silver_df
    .agg(F.sum("global_sales").alias("total"))
    .first()["total"]
)

for name, table_name in [
    ("YEAR", gold_year_table),
    ("PLATFORM", gold_platform_table),
    ("GENRE", gold_genre_table)
]:
    gold_sales = (
        spark.table(table_name)
        .agg(F.sum("global_sales").alias("total"))
        .first()["total"]
    )

    difference = abs(float(gold_sales) - float(source_global_sales))

    print(f"{name} sales difference: {difference:.2f}")

    assert difference <= 0.25, (
        f"{name} sales reconciliation failed"
    )

source_regional_sales = (
    silver_df
    .agg(
        F.sum("na_sales").alias("na"),
        F.sum("eu_sales").alias("eu"),
        F.sum("jp_sales").alias("jp"),
        F.sum("other_sales").alias("other")
    )
    .first()
)

expected_regional_total = sum(
    float(source_regional_sales[column] or 0)
    for column in ["na", "eu", "jp", "other"]
)

actual_regional_total = (
    spark.table(gold_region_table)
    .agg(F.sum("regional_sales").alias("total"))
    .first()["total"]
)

regional_difference = abs(
    float(actual_regional_total) - expected_regional_total
)

print(f"Regional sales difference: {regional_difference:.2f}")

assert regional_difference <= 0.03, (
    "Regional sales reconciliation failed"
)

print("\nGOLD LAYER VALIDATION PASSED")


In [0]:
%sql
SELECT *
FROM workspace.video_game_sales.gold_sales_by_platform
ORDER BY global_sales DESC
LIMIT 10;

In [0]:
%sql
SELECT *
FROM workspace.video_game_sales.gold_sales_by_genre
ORDER BY global_sales DESC;

In [0]:
%sql
SELECT *
FROM workspace.video_game_sales.gold_top_games
ORDER BY global_sales DESC;